## 1. Install dependencies

In [ ]:
!pip -q install -U transformers peft accelerate sentence-transformers faiss-cpu beautifulsoup4 requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 104.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## 2. Define a small starter set of public medical pages

In [ ]:
import requests, re, json, os, textwrap, time
from bs4 import BeautifulSoup
from pathlib import Path

# Start with a small, transparent set. You can add more pages after verifying URLs.
SOURCE_URLS = [
    "https://medlineplus.gov/highbloodpressure.html",
    "https://medlineplus.gov/diabetes.html",
    "https://medlineplus.gov/flu.html",
    "https://medlineplus.gov/heartdiseases.html",
    "https://www.cdc.gov/antibiotic-use/about/index.html",
    "https://www.who.int/news-room/fact-sheets/detail/healthy-diet",
]

HEADERS = {"User-Agent": "MedBotEducationalProject/1.0 (academic demo)"}
documents = []

for url in SOURCE_URLS:
    try:
        response = requests.get(url, headers=HEADERS, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header", "noscript"]):
            tag.decompose()
        title = soup.title.get_text(" ", strip=True) if soup.title else url
        main = soup.find("main") or soup.body or soup
        text = re.sub(r"\\s+", " ", main.get_text(" ", strip=True)).strip()
        if len(text) >= 500:
            documents.append({"source": url, "title": title, "text": text})
            print("OK:", title, "| chars:", len(text))
        else:
            print("Skipped short page:", url, "| chars:", len(text))
    except Exception as e:
        print("Could not fetch:", url, "|", type(e).__name__, str(e)[:180])

if not documents:
    raise RuntimeError(
        "No source pages were downloaded. Check the URLs/network, or manually upload "
        "plain-text medical documents and use them as documents."
    )

print("\\nDocuments collected:", len(documents))

OK: High Blood Pressure | Hypertension | MedlinePlus | chars: 11882
OK: Diabetes | Type 1 Diabetes | Type 2 Diabetes | MedlinePlus | chars: 14305
OK: Flu | Flu Symptoms | Stomach Flu | Influenza | MedlinePlus | chars: 10384
OK: Heart Diseases | Cardiac Diseases | MedlinePlus | chars: 13317
OK: Healthy Habits: Antibiotic Do's and Don'ts | Antibiotic Prescribing and Use | CDC | chars: 4100
OK: Healthy diet | chars: 21467
\nDocuments collected: 6


## 3. Chunk the documents

In [ ]:
CHUNK_WORDS = 180
OVERLAP_WORDS = 35

chunks = []
for doc_id, doc in enumerate(documents):
    words = doc["text"].split()
    start = 0
    while start < len(words):
        piece = words[start:start + CHUNK_WORDS]
        if len(piece) >= 35:
            chunks.append({
                "chunk_id": len(chunks),
                "doc_id": doc_id,
                "title": doc["title"],
                "source": doc["source"],
                "text": " ".join(piece)
            })
        if start + CHUNK_WORDS >= len(words):
            break
        start += CHUNK_WORDS - OVERLAP_WORDS

print("Total chunks:", len(chunks))
print("\\nExample chunk:")
print(json.dumps(chunks[0], indent=2, ensure_ascii=False)[:1500])

Total chunks: 80
\nExample chunk:
{
  "chunk_id": 0,
  "doc_id": 0,
  "title": "High Blood Pressure | Hypertension | MedlinePlus",
  "source": "https://medlineplus.gov/highbloodpressure.html",
  "text": "Skip navigation Official websites use .gov A .gov website belongs to an official government organization in the United States. Secure .gov websites use HTTPS A lock ( Lock Locked padlock icon ) or https:// means youâve safely connected to the .gov website. Share sensitive information only on official, secure websites. You Are Here: Home → Health Topics → High Blood Pressure URL of this page: https://medlineplus.gov/highbloodpressure.html High Blood Pressure Also called: Benign essential hypertension, Essential hypertension, HBP, HTN, Hypertension On this page Basics Summary Start Here Symptoms Diagnosis and Tests Prevention and Risk Factors Treatments and Therapies Learn More Living With Related Issues Specifics Genetics See, Play and Learn Videos and Tutorials Test Your Knowledge Re

## 4. Embed chunks and create the FAISS index

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)

texts = [c["text"] for c in chunks]
embeddings = embedder.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print("Embedding matrix shape:", embeddings.shape)
print("FAISS vectors indexed:", index.ntotal)

RAG_DIR = Path("/content/medbot_rag")
RAG_DIR.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(RAG_DIR / "medical.faiss"))

with open(RAG_DIR / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print("Index and chunks saved to:", RAG_DIR)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding matrix shape: (80, 384)
FAISS vectors indexed: 80
Index and chunks saved to: /content/medbot_rag


## 5. History-aware retrieval

In [ ]:
def retrieve_evidence(question, history=None, top_k=4):
    history = history or []
    # Use the current question plus a short summary of recent turns as retrieval context.
    recent = history[-4:]
    history_text = " ".join(
        f"{turn.get('role', 'user')}: {turn.get('content', '')}"
        for turn in recent
    )
    retrieval_query = (history_text + " " + question).strip()
    q_emb = embedder.encode(
        [retrieval_query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    scores, ids = index.search(q_emb, min(top_k, len(chunks)))

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        item = dict(chunks[int(idx)])
        item["similarity_score"] = float(score)
        results.append(item)
    return retrieval_query, results

test_question = "What are common ways to help manage high blood pressure?"
query_used, retrieved = retrieve_evidence(test_question)

print("Retrieval query:", query_used)
for rank, item in enumerate(retrieved, 1):
    print(f"\\n[{rank}] {item['title']} | score={item['similarity_score']:.3f}")
    print("Source:", item["source"])
    print(item["text"][:500], "...")

Retrieval query: What are common ways to help manage high blood pressure?
\n[1] High Blood Pressure | Hypertension | MedlinePlus | score=0.536
Source: https://medlineplus.gov/highbloodpressure.html
kind of blood pressure, it develops over time as you get older. Secondary high blood pressure. This is caused by another medical condition or use of certain medicines. It usually gets better after you treat that condition or stop taking the medicines that are causing it. Why do I need to worry about high blood pressure? When your blood pressure stays high over time, it causes the heart to pump harder and work overtime, possibly leading to serious health problems such as heart attack , stroke , h ...
\n[2] High Blood Pressure | Hypertension | MedlinePlus | score=0.471
Source: https://medlineplus.gov/highbloodpressure.html
Skip navigation Official websites use .gov A .gov website belongs to an official government organization in the United States. Secure .gov websites use HTTPS A lock ( Lock L

## 6. Load the fine-tuned adapter (if it is available in this runtime)

If your Colab session restarted, the adapter folder may not persist. Upload the adapter ZIP you downloaded in Phase 2, then set `ADAPTER_DIR` to the extracted folder. If unavailable, the notebook falls back to the base model and clearly labels that choice.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "/content/medbot_lora_adapter"

if not Path(ADAPTER_DIR).exists():
    print("Adapter folder not found in this runtime.")
    print("Upload and extract medbot_lora_adapter.zip, then update ADAPTER_DIR.")
    USE_ADAPTER = False
else:
    USE_ADAPTER = True

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

if USE_ADAPTER:
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    print("Loaded base model with MedBot LoRA adapter.")
else:
    model = base_model
    print("Loaded base model only; adapter is not active.")

model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Adapter folder not found in this runtime.
Upload and extract medbot_lora_adapter.zip, then update ADAPTER_DIR.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded base model only; adapter is not active.


## 7. Generate a grounded answer with source references

In [ ]:
def answer_with_rag(question, history=None, top_k=4, min_score=0.20, max_new_tokens=220):
    _, evidence = retrieve_evidence(question, history=history, top_k=top_k)

    # Similarity thresholds are heuristic, not calibrated confidence scores.
    usable = [e for e in evidence if e["similarity_score"] >= min_score]
    if not usable:
        return {
            "answer": (
                "I could not find sufficiently relevant evidence in the indexed sources. "
                "I cannot reliably answer from this knowledge base. Please consult a "
                "qualified healthcare professional for personal medical advice."
            ),
            "sources": evidence,
            "refused": True
        }

    context = "\\n\\n".join(
        f"[Source {i+1}] {e['title']}\\nURL: {e['source']}\\nEvidence: {e['text']}"
        for i, e in enumerate(usable)
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are MedBot, an educational medical information assistant. "
                "Use only the supplied evidence for medical factual claims. "
                "If evidence does not answer the question, state that limitation. "
                "Do not diagnose, prescribe, or invent drug doses. For urgent symptoms, "
                "recommend emergency medical care. Include source labels such as [Source 1]."
            )
        },
        {
            "role": "user",
            "content": (
                f"Question: {question}\\n\\n"
                f"Retrieved evidence:\\n{context}\\n\\n"
                "Give a concise, cautious answer with source labels."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3500)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = generated[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return {
        "answer": answer,
        "sources": usable,
        "refused": False
    }

result = answer_with_rag(
    "What are common ways to help manage high blood pressure?"
)

print("MEDBOT ANSWER:\\n")
print(result["answer"])
print("\\nRETRIEVED SOURCES:")
for i, src in enumerate(result["sources"], 1):
    print(f"[Source {i}] {src['title']} — {src['source']} (similarity={src['similarity_score']:.3f})")

MEDBOT ANSWER:\n
What are common ways to help manage high blood pressure? \n\nTreatments for high blood pressure include heart-healthy lifestyle changes and medicines. You will work with your provider to come up with a treatment plan. It may include only the lifestyle changes. These changes, such as heart-healthy eating and exercise, can be very effective. But sometimes the changes do not control or lower your high blood pressure. Then you may need to take medicine. There are different types of blood pressure medicines. Some people need to take more than one type of medicine. Lifestyle Changes The best way to prevent high blood pressure is through healthy eating, regular physical activity, maintaining a healthy weight, and avoiding tobacco use. Avoiding tobacco use includes quitting smoking and reducing exposure to secondhand smoke. Regular Physical Activity Regular physical activity helps keep your blood pressure within normal range. Aim for at least 150 minutes of moderate aerobic ac

## 8. Test a follow-up question using conversation history

In [ ]:
history = [
    {"role": "user", "content": "What is high blood pressure?"},
    {"role": "assistant", "content": "It is a condition in which blood pressure is persistently elevated."}
]

followup = answer_with_rag(
    "What lifestyle changes may help?",
    history=history
)

print("FOLLOW-UP ANSWER:\\n")
print(followup["answer"])
print("\\nSOURCES:")
for src in followup["sources"]:
    print("-", src["title"], src["source"])

FOLLOW-UP ANSWER:\n
Lifestyle changes that may help manage high blood pressure include:

1. **Heart-Healthy Eating**: Adopting a diet rich in fruits, vegetables, whole grains, lean proteins, and low-fat dairy products while limiting sodium intake.
   
   Evidence Source: [Source 1]

2. **Regular Exercise**: Engaging in physical activity regularly, aiming for at least 150 minutes per week of moderate aerobic activity or 75 minutes per week of vigorous activity.

   Evidence Source: [Source 1]

3. **Weight Management**: Maintaining a healthy weight through balanced nutrition and regular physical activity.

   Evidence Source: [Source 1]

4. **Limit Alcohol Consumption**: Drinking alcohol in moderation, typically no more than one drink per day for women and two drinks per day for men.

   Evidence Source: [Source 1]

5. **Stop Smoking**: Quitting smoking significantly reduces the risk of developing high blood pressure and improves overall cardiovascular health.

   Evidence Source: [Sourc

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving medbot_lora_adapter.zip to medbot_lora_adapter.zip


In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = "/content/medbot_lora_adapter.zip"
EXTRACT_DIR = Path("/content/medbot_adapter_extracted")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

# Find the actual adapter folder, even if the ZIP contains a nested directory.
configs = list(EXTRACT_DIR.rglob("adapter_config.json"))

if not configs:
    raise FileNotFoundError(
        "adapter_config.json was not found. Check that you uploaded the correct ZIP."
    )

ADAPTER_DIR = str(configs[0].parent)

print("Adapter folder:", ADAPTER_DIR)
print("Adapter files:")
for p in sorted(Path(ADAPTER_DIR).iterdir()):
    print("-", p.name)

Adapter folder: /content/medbot_adapter_extracted
Adapter files:
- README.md
- adapter_config.json
- adapter_model.safetensors
- chat_template.jinja
- tokenizer.json
- tokenizer_config.json


In [ ]:
!pip install -q -U torchao

import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Release the previous base model to free GPU memory.
try:
    del model
except NameError:
    pass

try:
    del base_model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False
)

model.eval()

print("SUCCESS: Fine-tuned MedBot adapter loaded.")
print("Model:", BASE_MODEL)
print("Adapter active:", model.active_adapter)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 32.7 MB/s eta 0:00:00


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

SUCCESS: Fine-tuned MedBot adapter loaded.
Model: Qwen/Qwen2.5-1.5B-Instruct
Adapter active: default


In [ ]:
!pip -q install requests gradio

In [ ]:
def calculate_bmi(weight_kg, height_cm):
    """Calculate BMI from weight in kg and height in cm."""
    try:
        weight_kg = float(weight_kg)
        height_cm = float(height_cm)

        if weight_kg <= 0 or height_cm <= 0:
            return {"error": "Weight and height must be positive."}

        height_m = height_cm / 100
        bmi = weight_kg / (height_m ** 2)

        if bmi < 18.5:
            category = "Underweight"
        elif bmi < 25:
            category = "Normal weight"
        elif bmi < 30:
            category = "Overweight"
        else:
            category = "Obesity"

        return {
            "bmi": round(bmi, 2),
            "category": category,
            "note": (
                "BMI is a screening measure, not a diagnosis. "
                "Interpretation depends on individual circumstances."
            )
        }

    except (ValueError, TypeError):
        return {"error": "Please provide valid numeric values."}


# Test
print(calculate_bmi(70, 175))

{'bmi': 22.86, 'category': 'Normal weight', 'note': 'BMI is a screening measure, not a diagnosis. Interpretation depends on individual circumstances.'}


In [ ]:
import requests

def get_drug_info(drug_name):
    """Search openFDA for a drug label."""
    if not drug_name or not drug_name.strip():
        return {"error": "Enter a medication name."}

    url = "https://api.fda.gov/drug/label.json"
    params = {
        "search": f'openfda.brand_name:"{drug_name.strip()}"',
        "limit": 1
    }

    try:
        response = requests.get(url, params=params, timeout=20)

        if response.status_code == 404:
            return {
                "message": "No matching label was found.",
                "drug": drug_name
            }

        response.raise_for_status()
        data = response.json()
        result = data["results"][0]
        openfda = result.get("openfda", {})

        def first_text(field):
            values = result.get(field, [])
            return values[0] if values else "Not available in this label."

        return {
            "drug": drug_name,
            "brand_name": openfda.get("brand_name", [drug_name])[0],
            "generic_name": openfda.get("generic_name", ["Not listed"])[0],
            "purpose": first_text("purpose"),
            "indications_and_usage": first_text("indications_and_usage"),
            "warnings": first_text("warnings"),
            "source": "openFDA drug labeling API",
            "disclaimer": (
                "Label information is not personalized medical advice. "
                "Confirm medication decisions with a qualified professional."
            )
        }

    except requests.RequestException as exc:
        return {"error": f"API request failed: {exc}"}
    except (KeyError, IndexError, ValueError):
        return {"error": "The API response did not contain a usable drug label."}


# Test with a common medication name
drug_result = get_drug_info("ibuprofen")
print(drug_result)

{'drug': 'ibuprofen', 'brand_name': 'Ibuprofen Dye Free', 'generic_name': 'IBUPROFEN', 'purpose': 'Purpose Pain reliever/fever reducer', 'indications_and_usage': 'Uses temporarily relieves minor aches and pains due to: headache toothache backache menstrual cramps the common cold muscular aches minor pain of arthritis temporarily reduces fever', 'warnings': "Warnings Allergy alert: Ibuprofen may cause a severe allergic reaction, especially in people allergic to aspirin. Symptoms may include: rash facial swelling asthma (wheezing) hives skin reddening shock blisters If an allergic reaction occurs, stop use and seek medical help right away. Stomach bleeding warning: This product contains an NSAID, which may cause severe stomach bleeding. The chance is higher if you: take more or for a longer time than directed take a blood thinning (anticoagulant) or steroid drug have had stomach ulcers or bleeding problems have 3 or more alcoholic drinks every day while using this product are age 60 or o

In [ ]:
print("BMI TOOL TEST")
print(calculate_bmi(70, 175))

print("\nDRUG INFORMATION TOOL TEST")
print(get_drug_info("ibuprofen"))

BMI TOOL TEST
{'bmi': 22.86, 'category': 'Normal weight', 'note': 'BMI is a screening measure, not a diagnosis. Interpretation depends on individual circumstances.'}

DRUG INFORMATION TOOL TEST
{'drug': 'ibuprofen', 'brand_name': 'Ibuprofen Dye Free', 'generic_name': 'IBUPROFEN', 'purpose': 'Purpose Pain reliever/fever reducer', 'indications_and_usage': 'Uses temporarily relieves minor aches and pains due to: headache toothache backache menstrual cramps the common cold muscular aches minor pain of arthritis temporarily reduces fever', 'warnings': "Warnings Allergy alert: Ibuprofen may cause a severe allergic reaction, especially in people allergic to aspirin. Symptoms may include: rash facial swelling asthma (wheezing) hives skin reddening shock blisters If an allergic reaction occurs, stop use and seek medical help right away. Stomach bleeding warning: This product contains an NSAID, which may cause severe stomach bleeding. The chance is higher if you: take more or for a longer time t

In [ ]:
!pip -q install -U gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 9.1 MB/s eta 0:00:00


In [ ]:
import gradio as gr
import json

def medbot_reply(message, mode, weight, height, history):
    message = (message or "").strip()
    history = history or []

    if not message:
        return history, ""

    try:
        if mode == "BMI Calculator":
            result = calculate_bmi(weight, height)
            reply = json.dumps(result, indent=2, ensure_ascii=False)

        elif mode == "Drug Information":
            result = get_drug_info(message)
            reply = json.dumps(result, indent=2, ensure_ascii=False)

        else:
            # Use recent conversation turns for history-aware retrieval.
            recent_history = history[-6:]
            result = answer_with_rag(
                message,
                history=recent_history,
                top_k=4
            )

            reply = result["answer"]

            if result.get("sources"):
                reply += "\n\n**Retrieved sources:**\n"
                for i, source in enumerate(result["sources"], 1):
                    reply += (
                        f"\n{i}. [{source['title']}]"
                        f"({source['source']})"
                    )

            reply += (
                "\n\n*Educational information only. "
                "This is not a diagnosis or a substitute for "
                "professional medical advice.*"
            )

    except Exception as exc:
        reply = (
            "Sorry, this request could not be completed. "
            "Please try again. Technical detail: "
            + str(exc)[:250]
        )

    # In older Gradio versions, history must be a list of lists: [[user_msg, bot_msg], ...]
    history = history + [[message, reply]]

    return history, ""


with gr.Blocks(title="MedBot — Medical Q&A Assistant") as demo:
    gr.Markdown(
        """
        # MedBot 🩺
        ### Medical Question Answering Assistant

        Ask medical information questions, retrieve evidence,
        calculate BMI, or look up publicly available drug labels.

        **Disclaimer:** Educational prototype only. Not for diagnosis,
        emergency triage, or personalized medication decisions.
        """
    )

    mode = gr.Radio(
        choices=[
            "Medical Q&A (RAG)",
            "BMI Calculator",
            "Drug Information"
        ],
        value="Medical Q&A (RAG)",
        label="Choose a function"
    )

    with gr.Row():
        weight = gr.Number(
            label="Weight (kg) — for BMI",
            value=70
        )
        height = gr.Number(
            label="Height (cm) — for BMI",
            value=175
        )

    chatbot = gr.Chatbot(
        label="MedBot Conversation",
        height=420
    )

    message = gr.Textbox(
        label="Your question or medication name",
        placeholder=(
            "Ask a medical question, enter a drug name, "
            "or describe what you want to calculate..."
        )
    )

    with gr.Row():
        submit = gr.Button("Send", variant="primary")
        clear = gr.Button("Clear conversation")

    submit.click(
        fn=medbot_reply,
        inputs=[message, mode, weight, height, chatbot],
        outputs=[chatbot, message]
    )

    message.submit(
        fn=medbot_reply,
        inputs=[message, mode, weight, height, chatbot],
        outputs=[chatbot, message]
    )

    clear.click(
        fn=lambda: ([], ""),
        outputs=[chatbot, message]
    )

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://259a920f59e127aa6b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

def medbot_reply(message, mode, weight, height, history):
    import json

    message = (message or "").strip()

    # Keep only valid Gradio messages; discard old tuple-format history.
    clean_history = []
    for item in (history or []):
        if (
            isinstance(item, dict)
            and item.get("role") in ("user", "assistant", "system")
            and isinstance(item.get("content"), str)
        ):
            clean_history.append({
                "role": item["role"],
                "content": item["content"]
            })

    if not message:
        return clean_history, ""

    try:
        if mode == "BMI Calculator":
            reply = json.dumps(
                calculate_bmi(weight, height),
                indent=2,
                ensure_ascii=False
            )

        elif mode == "Drug Information":
            reply = json.dumps(
                get_drug_info(message),
                indent=2,
                ensure_ascii=False
            )

        else:
            result = answer_with_rag(
                message,
                history=clean_history[-6:],
                top_k=4
            )
            reply = result["answer"]

            if result.get("sources"):
                reply += "\n\nRetrieved sources:\n"
                for i, source in enumerate(result["sources"], 1):
                    reply += (
                        f"\n[{i}] {source['title']}: "
                        f"{source['source']}"
                    )

            reply += (
                "\n\nEducational prototype only; "
                "not a diagnosis or substitute for professional care."
            )

    except Exception as exc:
        reply = f"Request failed: {type(exc).__name__}: {str(exc)[:200]}"

    clean_history.extend([
        {"role": "user", "content": message},
        {"role": "assistant", "content": reply}
    ])

    return clean_history, ""


In [ ]:
chatbot = gr.Chatbot(
    label="MedBot Conversation",
    height=420
)

In [ ]:
import gradio as gr
import inspect

print(inspect.signature(gr.Chatbot))

(value: 'list[MessageDict | Message] | Callable | None' = None, *, label: 'str | I18nData | None' = None, every: 'Timer | float | None' = None, inputs: 'Component | Sequence[Component] | set[Component] | None' = None, show_label: 'bool | None' = None, container: 'bool' = True, scale: 'int | None' = None, min_width: 'int' = 160, visible: "bool | Literal['hidden']" = True, elem_id: 'str | None' = None, elem_classes: 'list[str] | str | None' = None, autoscroll: 'bool' = True, render: 'bool' = True, key: 'int | str | tuple[int | str, ...] | None' = None, preserved_by_key: 'list[str] | str | None' = 'value', height: 'int | str | None' = 400, resizable: 'bool' = False, max_height: 'int | str | None' = None, min_height: 'int | str | None' = None, editable: "Literal['user', 'all'] | None" = None, latex_delimiters: 'list[dict[str, str | bool]] | None' = None, rtl: 'bool' = False, buttons: "list[Literal['share', 'copy', 'copy_all'] | Button] | None" = None, watermark: 'str | None' = None, avatar

In [ ]:
[
    {"role": "user", "content": "What is high blood pressure?"},
    {"role": "assistant", "content": "High blood pressure ..."}
]

[{'role': 'user', 'content': 'What is high blood pressure?'},
 {'role': 'assistant', 'content': 'High blood pressure ...'}]

In [ ]:
print("Gradio version:", gr.__version__)
print("Empty chatbot value:", chatbot.get_config().get("value"))
print("Callback test:", medbot_reply(
    "Hello",
    "Medical Q&A (RAG)",
    70,
    175,
    []
))

Gradio version: 6.30.0
Empty chatbot value: []
Callback test: ([{'role': 'user', 'content': 'Hello'}, {'role': 'assistant', 'content': 'I could not find sufficiently relevant evidence in the indexed sources. I cannot reliably answer from this knowledge base. Please consult a qualified healthcare professional for personal medical advice.\n\nRetrieved sources:\n\n[1] Healthy diet: https://www.who.int/news-room/fact-sheets/detail/healthy-diet\n[2] Healthy diet: https://www.who.int/news-room/fact-sheets/detail/healthy-diet\n[3] Diabetes | Type 1 Diabetes | Type 2 Diabetes | MedlinePlus: https://medlineplus.gov/diabetes.html\n[4] Healthy diet: https://www.who.int/news-room/fact-sheets/detail/healthy-diet\n\nEducational prototype only; not a diagnosis or substitute for professional care.'}], '')


In [ ]:

import gradio as gr

with gr.Blocks(title="MedBot Medical Q&A Assistant") as demo:
    gr.Markdown(
        """
        # 🩺 MedBot — Medical Q&A Assistant
        Ask medical questions, calculate BMI, or look up drug labels.

        **Disclaimer:** Educational prototype only. Not for diagnosis
        or personalized medication advice.
        """
    )

    mode = gr.Radio(
        choices=[
            "Medical Q&A (RAG)",
            "BMI Calculator",
            "Drug Information"
        ],
        value="Medical Q&A (RAG)",
        label="Choose a function"
    )

    with gr.Row():
        weight = gr.Number(label="Weight (kg)", value=70)
        height = gr.Number(label="Height (cm)", value=175)

    chatbot = gr.Chatbot(
        label="MedBot Conversation",
        height=420
    )

    message = gr.Textbox(
        label="Your question or medication name",
        placeholder="Ask a medical question..."
    )

    with gr.Row():
        send = gr.Button("Send", variant="primary")
        clear = gr.Button("Clear conversation")

    send.click(
        fn=medbot_reply,
        inputs=[message, mode, weight, height, chatbot],
        outputs=[chatbot, message]
    )

    message.submit(
        fn=medbot_reply,
        inputs=[message, mode, weight, height, chatbot],
        outputs=[chatbot, message]
    )

    clear.click(
        fn=lambda: ([], ""),
        inputs=None,
        outputs=[chatbot, message]
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c79d6aa824d2e16bdd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
